---
title: Jupyter Notebook Tutorial
short_title: Jupyter Tutorial
---

Pages can also be built from Jupyter Notebooks.
How cool is that?

:::{note}
These notebooks could be executed before the book is built, but that requires a comprehensive Python environment. If they are **not** executed, it makes sense to store the cell outputs.
:::

Here's a code cell and its output:

In [ ]:
import numpy as np

np.zeros(3)

In [ ]:
# Interactive time series: global surface temperature (NASA GISTEMP) vs. CO2 (NOAA Mauna Loa).
# One panel: light lines = raw monthly data (always shown), solid lines = smoothed (slider).
# Requires: pip install pandas bokeh   (or run `%pip install pandas bokeh` in a cell first)
import pandas as pd
from bokeh.io import output_notebook, show
from bokeh.layouts import column
from bokeh.models import (
    BoxAnnotation,
    ColumnDataSource,
    CustomJS,
    HoverTool,
    LinearAxis,
    Range1d,
    Slider,
)
from bokeh.plotting import figure

output_notebook()

GISTEMP_URL = "https://data.giss.nasa.gov/gistemp/tabledata_v4/GLB.Ts+dSST.csv"
MAUNA_LOA_URL = "https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_mlo.csv"
TEMP_COLOR, CO2_COLOR = "#d62728", "#1f77b4"  # smoothed (solid)
TEMP_RAW_COLOR, CO2_RAW_COLOR = "#f4a3a3", "#9cc5e8"  # raw (light red / light blue)
INITIAL_WINDOW = 12


def load_temperature() -> pd.DataFrame:
    """NASA GISTEMP: wide table (one row per year, Jan..Dec) -> monthly series."""
    raw = pd.read_csv(GISTEMP_URL, skiprows=1, na_values="***")
    months = [
        "Jan",
        "Feb",
        "Mar",
        "Apr",
        "May",
        "Jun",
        "Jul",
        "Aug",
        "Sep",
        "Oct",
        "Nov",
        "Dec",
    ]
    long = raw.melt(
        id_vars="Year", value_vars=months, var_name="month", value_name="temp"
    )
    long["date"] = pd.to_datetime(
        long["Year"].astype(str) + "-" + long["month"], format="%Y-%b"
    )
    return (
        long.dropna(subset=["temp"])
        .sort_values("date")[["date", "temp"]]
        .reset_index(drop=True)
    )


def load_co2() -> pd.DataFrame:
    """NOAA Mauna Loa monthly CO2 (comment lines start with '#'; -99.99 = missing)."""
    raw = pd.read_csv(MAUNA_LOA_URL, comment="#", na_values=[-99.99, -9.99])
    raw.columns = [c.strip() for c in raw.columns]
    raw["date"] = pd.to_datetime({"year": raw["year"], "month": raw["month"], "day": 1})
    return (
        raw.rename(columns={"average": "co2"})[["date", "co2"]]
        .dropna()
        .reset_index(drop=True)
    )


def make_source(dates, values) -> ColumnDataSource:
    """'raw' is the original series; 'y' is the smoothed one (rewritten by the slider)."""
    smoothed = values.rolling(INITIAL_WINDOW, center=True, min_periods=1).mean()
    return ColumnDataSource({"date": dates, "raw": values, "y": smoothed})


temp, co2 = load_temperature(), load_co2()
temp_src = make_source(temp["date"], temp["temp"])
co2_src = make_source(co2["date"], co2["co2"])

p = figure(
    title="Global surface temperature and atmospheric CO₂ (Mauna Loa)",
    x_axis_type="datetime",
    width=1200,
    height=400,
    sizing_mode="scale_width",  # 3:1
    tools="pan,wheel_zoom,box_zoom,reset,save",
    active_scroll="wheel_zoom",
    y_range=Range1d(temp["temp"].min() - 0.1, temp["temp"].max() + 0.1),
)
p.yaxis.axis_label = "Temperature anomaly (°C, vs 1951–1980)"
p.yaxis.axis_label_text_color = TEMP_COLOR

p.extra_y_ranges = {"co2": Range1d(co2["co2"].min() - 5, co2["co2"].max() + 5)}
p.add_layout(
    LinearAxis(
        y_range_name="co2", axis_label="CO₂ (ppm)", axis_label_text_color=CO2_COLOR
    ),
    "right",
)

# Raw lines first (light, underneath), smoothed lines on top (solid)
p.line(
    "date",
    "raw",
    source=temp_src,
    color=TEMP_RAW_COLOR,
    line_width=1.5,
    legend_label="Temperature, raw (NASA GISTEMP)",
)
p.line(
    "date",
    "raw",
    source=co2_src,
    color=CO2_RAW_COLOR,
    line_width=1.5,
    y_range_name="co2",
    legend_label="CO₂, raw (NOAA Mauna Loa)",
)
t_smooth = p.line(
    "date",
    "y",
    source=temp_src,
    color=TEMP_COLOR,
    line_width=2.5,
    legend_label="Temperature, smoothed",
)
c_smooth = p.line(
    "date",
    "y",
    source=co2_src,
    color=CO2_COLOR,
    line_width=2.5,
    y_range_name="co2",
    legend_label="CO₂, smoothed",
)

# Reference range
temp_ref_box = BoxAnnotation(
    left=pd.Timestamp("1951-01-01"),
    right=pd.Timestamp("1979-12-31"),
    fill_color="#C0C0C0",
    level="underlay",
)
p.add_layout(temp_ref_box)

p.add_tools(
    HoverTool(
        renderers=[t_smooth],
        mode="vline",
        tooltips=[
            ("Date", "@date{%b %Y}"),
            ("Smoothed", "@y{+0.00} °C"),
            ("Raw", "@raw{+0.00} °C"),
        ],
        formatters={"@date": "datetime"},
    ),
    HoverTool(
        renderers=[c_smooth],
        mode="vline",
        tooltips=[
            ("Date", "@date{%b %Y}"),
            ("Smoothed", "@y{0.0} ppm"),
            ("Raw", "@raw{0.0} ppm"),
        ],
        formatters={"@date": "datetime"},
    ),
)
p.legend.location = "top_left"
p.legend.click_policy = "hide"  # click a legend entry to toggle that series

# Smoothing slider (centered rolling mean of the raw data, computed in the browser)
smooth = Slider(
    start=1,
    end=60,
    value=INITIAL_WINDOW,
    step=1,
    title="Smoothing window (months; 1 = no smoothing)",
)
smooth.js_on_change(
    "value",
    CustomJS(
        args={"srcs": [temp_src, co2_src]},
        code="""
    const w = cb_obj.value, half = Math.floor(w / 2);
    for (const src of srcs) {
        const raw = src.data['raw'], n = raw.length, out = new Array(n);
        for (let i = 0; i < n; i++) {
            let s = 0, c = 0;
            for (let j = Math.max(0, i - half); j <= Math.min(n - 1, i + half); j++) {
                if (!isNaN(raw[j])) { s += raw[j]; c++; }
            }
            out[i] = c ? s / c : NaN;
        }
        src.data['y'] = out;
        src.change.emit();
    }
""",
    ),
)

show(column(p, smooth, sizing_mode="scale_width"))